In [3]:
# ==========================
# === STATE MACHINE TIMELINE PIPELINE (CONSTANT-STATE SEGMENTS) ===
#
# FIXES APPLIED (Mixed -> Single / "pure remove" confusion):
#   1) Episodes are still defined as a constant active-style SET, serialized as "A+B" (env_styles).
#   2) Episode boundary labels (boundary_event_types in List_Change_Episodes.csv) are NO LONGER
#      taken from the episode's start_event/end_event types (which can misleadingly be "removed").
#      Instead, they are derived from STATE DIFFERENCE between consecutive kept episodes:
#        - added(X+Y) and/or removed(X+Y)
#      This reflects reality: e.g., "Emu_Community+ThirdParty -> ThirdParty" becomes removed(Emu_Community),
#      not a "removed-started" episode.
#   3) ThirdParty remains a fully valid style and can appear as a single-style state ("ThirdParty").
#
# INPUT:
#   - Miner JSON files: *.emulator_timeline*.json
#
# OUTPUTS (under OBS_OUTPUT):
#   1) List_Change_Episodes.csv
#   2) List_Boundary_Commit_Events.csv
#   3) Repo_level_episodes.csv
#
# ==========================

from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Set
from collections import defaultdict

import pandas as pd


# ------------------------------------------------------------------
# Paths (EDIT)
# ------------------------------------------------------------------
WORK_ROOT   = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2")
MINE_DIR    = WORK_ROOT / "Mine_Full"     # where your *.json miner outputs are

OBS_OUTPUT  = WORK_ROOT / "Observations"
OBS_OUTPUT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# Settings
# ------------------------------------------------------------------
CUTOFF_ISO   = "2025-08-10 23:59:59 +0000"
PERSIST_DAYS = 14.0

# If your all-branch miner still writes on_default flags and you want to keep ONLY on_default=1,
# set this True. For all-branches timelines, typically leave False.
USE_ON_DEFAULT_FILTER = False

# If multiple JSONs exist per repo, select best by highest count of effective events (tie by mtime).
DEDUP_BY_REPO = True

STYLE_ORDER: List[str] = ["Emu_Custom", "Emu_Community", "GMD", "ThirdParty"]
STYLE_RANK: Dict[str, int] = {s: i for i, s in enumerate(STYLE_ORDER)}
CANONICAL_STYLES: Set[str] = set(STYLE_ORDER)


# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def parse_iso(s: Optional[str]) -> Optional[datetime]:
    if not s:
        return None
    s = str(s).strip()
    if s.endswith("Z"):
        s = s[:-1] + "+00:00"
    # normalize +0000 -> +00:00
    if len(s) >= 5 and (s[-5] in ["+", "-"]) and s[-3] != ":":
        s = s[:-2] + ":" + s[-2:]
    try:
        dt = datetime.fromisoformat(s)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    except Exception:
        return None

CUTOFF_DT = parse_iso(CUTOFF_ISO)
assert CUTOFF_DT is not None, f"Bad CUTOFF_ISO: {CUTOFF_ISO}"

def iso(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat()

def days_between(a: datetime, b: datetime) -> float:
    return (b - a).total_seconds() / 86400.0

def repo_from_filename(jf: Path) -> str:
    nm = jf.name
    if ".emulator_timeline" in nm:
        return nm.split(".emulator_timeline", 1)[0]
    return jf.stem

def find_json_files(root: Path) -> List[Path]:
    files = sorted(root.rglob("*.json"))
    out: List[Path] = []
    for f in files:
        n = f.name.lower()
        if ".emulator_timeline" not in n:
            continue
        if n.startswith("index_summary") or "index_summary" in n:
            continue
        out.append(f)
    return sorted(out)

def fmt_state(active: Set[str]) -> Tuple[str, ...]:
    ordered = [s for s in STYLE_ORDER if s in active]
    extras = sorted([s for s in active if s not in CANONICAL_STYLES])
    return tuple(ordered + extras)

def state_str(st: Tuple[str, ...]) -> str:
    return "+".join(st)

def env_style_mixed_from_state(st: Tuple[str, ...]) -> str:
    if not st:
        return ""
    return "Mixed" if len(st) >= 2 else st[0]

def boundary_change_label(cur: Tuple[str, ...], nxt: Tuple[str, ...]) -> str:
    """Describe the transition as added()/removed() based on state set difference."""
    cur_set = set(cur)
    nxt_set = set(nxt)

    added = [s for s in STYLE_ORDER if s in (nxt_set - cur_set)]
    removed = [s for s in STYLE_ORDER if s in (cur_set - nxt_set)]

    parts: List[str] = []
    if added:
        parts.append("added(" + "+".join(added) + ")")
    if removed:
        parts.append("removed(" + "+".join(removed) + ")")

    return " & ".join(parts) if parts else "nochange"


# ------------------------------------------------------------------
# Read JSONs (+ optional repo dedup)
# ------------------------------------------------------------------
json_files = find_json_files(MINE_DIR)
if not json_files:
    raise FileNotFoundError(f"No miner JSON files found under: {MINE_DIR}")

loaded: List[Tuple[Path, Dict]] = []
for jf in json_files:
    try:
        with jf.open("r", encoding="utf-8") as f:
            rec = json.load(f)
        loaded.append((jf, rec))
    except Exception:
        continue

if not loaded:
    raise RuntimeError(f"Could not load any valid miner JSON under: {MINE_DIR}")

def count_raw_events(rec: Dict) -> int:
    events = rec.get("events") or {}
    c = 0
    for s in STYLE_ORDER:
        c += len(events.get(s, []) or [])
    return c

selected: List[Tuple[Path, Dict]] = []
if not DEDUP_BY_REPO:
    selected = loaded
else:
    best_by_repo: Dict[str, Tuple[int, float, Path, Dict]] = {}  # repo -> (score, mtime, path, rec)
    for jf, rec in loaded:
        repo = rec.get("repo_name") or repo_from_filename(jf)
        score = count_raw_events(rec)
        try:
            mtime = jf.stat().st_mtime
        except Exception:
            mtime = 0.0
        cur = best_by_repo.get(repo)
        if (cur is None) or (score > cur[0]) or (score == cur[0] and mtime > cur[1]):
            best_by_repo[repo] = (score, mtime, jf, rec)
    selected = [(v[2], v[3]) for v in best_by_repo.values()]
    selected.sort(key=lambda t: (str((t[1].get("repo_name") or repo_from_filename(t[0]))).lower(),
                                 t[0].name.lower()))

print(f"[info] Found {len(json_files)} candidate JSON files under Mine dir.")
print(f"[info] Loaded {len(loaded)} JSON files successfully.")
print(f"[info] Selected {len(selected)} JSON records for processing (dedup_by_repo={DEDUP_BY_REPO}).")


# ------------------------------------------------------------------
# EFFECTIVE events (only those that flip active status)
#   Ordering: (dt, commit, removed-before-added, style_rank)
# ------------------------------------------------------------------
def collect_effective_events(rec: Dict, cutoff_dt: datetime) -> List[Dict]:
    events_dict = rec.get("events") or {}
    raw: List[Dict] = []

    for style in STYLE_ORDER:
        for ev in (events_dict.get(style, []) or []):
            et = str(ev.get("event") or "").strip().lower()
            if et not in {"added", "removed"}:
                continue

            dt = parse_iso(ev.get("date"))
            if dt is None or dt > cutoff_dt:
                continue

            sha = (ev.get("commit") or "").strip()
            if not sha:
                continue

            if USE_ON_DEFAULT_FILTER:
                on_default = ev.get("on_default", 1)
                if str(on_default) != "1":
                    continue

            raw.append({
                "dt": dt,
                "commit": sha,
                "style": style,
                "etype": et,
            })

    raw.sort(key=lambda e: (
        e["dt"],
        e["commit"],
        0 if e["etype"] == "removed" else 1,
        STYLE_RANK.get(e["style"], 99),
    ))

    active: Set[str] = set()
    effective: List[Dict] = []

    for e in raw:
        s = e["style"]
        et = e["etype"]
        etype_order = 0 if et == "removed" else 1
        key = (e["dt"], e["commit"], etype_order, STYLE_RANK.get(s, 99))

        if et == "added":
            if s not in active:
                active.add(s)
                effective.append({**e, "key": key})
        else:  # removed
            if s in active:
                active.remove(s)
                effective.append({**e, "key": key})

    return effective


# ------------------------------------------------------------------
# Build raw constant-state segments (state machine)
#   Each segment begins AFTER applying its boundary event.
# ------------------------------------------------------------------
def build_raw_state_segments(all_effective: List[Dict], cutoff_dt: datetime) -> List[Dict]:
    if not all_effective:
        return []

    active: Set[str] = set()
    segments: List[Dict] = []

    cur_state: Optional[Tuple[str, ...]] = None
    cur_start_dt: Optional[datetime] = None
    cur_start_event: Optional[Dict] = None

    for ev in all_effective:
        dt = ev["dt"]
        sty = ev["style"]
        ety = ev["etype"]

        # apply event to active set
        if ety == "added":
            active.add(sty)
        else:
            active.discard(sty)

        if not active:
            # close existing segment (if any) into empty; we do not keep empty segments
            if cur_state is not None and cur_start_dt is not None:
                segments.append({
                    "state": cur_state,
                    "start_dt": cur_start_dt,
                    "end_dt": dt,
                    "start_event": cur_start_event,  # event that created this state
                    "end_event": ev,                 # event that ended it (or emptied it)
                })
                cur_state, cur_start_dt, cur_start_event = None, None, None
            continue

        new_state = fmt_state(active)

        if cur_state is None:
            cur_state = new_state
            cur_start_dt = dt
            cur_start_event = ev
        elif new_state != cur_state:
            # close previous at boundary dt
            segments.append({
                "state": cur_state,
                "start_dt": cur_start_dt,
                "end_dt": dt,
                "start_event": cur_start_event,
                "end_event": ev,
            })
            # open new
            cur_state = new_state
            cur_start_dt = dt
            cur_start_event = ev

    # close to cutoff
    if cur_state is not None and cur_start_dt is not None:
        segments.append({
            "state": cur_state,
            "start_dt": cur_start_dt,
            "end_dt": cutoff_dt,
            "start_event": cur_start_event,
            "end_event": None,
        })

    return segments


# ------------------------------------------------------------------
# Drop invalid short-lived segments (<persist_days) WITHOUT creating boundaries:
#   - remove them entirely
#   - bridge: previous kept segment ends at the next kept segment start
# ------------------------------------------------------------------
def collapse_invalid_segments(segments: List[Dict], cutoff_dt: datetime, persist_days: float) -> List[Dict]:
    if not segments:
        return []

    def is_valid(seg: Dict) -> bool:
        dur = days_between(seg["start_dt"], seg["end_dt"])
        return (dur >= persist_days) or (seg["end_dt"] == cutoff_dt)

    kept: List[Dict] = []
    cur: Optional[Dict] = None

    for seg in segments:
        if not is_valid(seg):
            continue

        if cur is None:
            cur = dict(seg)
            continue

        # bridge across invalid segments: close cur at the start of seg
        cur["end_dt"] = seg["start_dt"]
        cur["end_event"] = seg.get("start_event")  # boundary event that starts next kept state

        cur["duration_days"] = days_between(cur["start_dt"], cur["end_dt"])
        cur["is_active_at_cutoff"] = 1 if (cur["end_dt"] == cutoff_dt) else 0
        kept.append(cur)

        cur = dict(seg)

    if cur is not None:
        cur["duration_days"] = days_between(cur["start_dt"], cur["end_dt"])
        cur["is_active_at_cutoff"] = 1 if (cur["end_dt"] == cutoff_dt) else 0
        kept.append(cur)

    # merge adjacent identical states
    merged: List[Dict] = []
    for s in kept:
        if merged and merged[-1]["state"] == s["state"] and merged[-1]["end_dt"] == s["start_dt"]:
            merged[-1]["end_dt"] = s["end_dt"]
            merged[-1]["end_event"] = s.get("end_event")
            merged[-1]["duration_days"] = days_between(merged[-1]["start_dt"], merged[-1]["end_dt"])
            merged[-1]["is_active_at_cutoff"] = s["is_active_at_cutoff"]
        else:
            merged.append(s)

    return merged


# ------------------------------------------------------------------
# Main pipeline
# ------------------------------------------------------------------
episode_rows: List[Dict] = []
boundary_event_rows: List[Dict] = []
repo_level_rows: List[Dict] = []

for jf, rec in selected:
    repo = rec.get("repo_name") or repo_from_filename(jf)
    if not repo:
        continue

    repo_name = repo
    full_name = repo.replace("__", ".")
    timeline_scope = str(rec.get("timeline_scope", "") or "")
    qa_issue = str(rec.get("qa_issue", "") or "")

    all_effective = collect_effective_events(rec, CUTOFF_DT)
    raw_segments = build_raw_state_segments(all_effective, CUTOFF_DT)
    kept_segments = collapse_invalid_segments(raw_segments, CUTOFF_DT, PERSIST_DAYS)

    # Convert kept segments into episode-like rows
    kept_eps: List[Dict] = []
    for idx, seg in enumerate(kept_segments, start=1):
        st: Tuple[str, ...] = seg["state"]
        st_s = state_str(st)

        next_state: Optional[Tuple[str, ...]] = None
        next_state_s = ""
        if idx < len(kept_segments):
            next_state = kept_segments[idx]["state"]  # idx is 1-based
            next_state_s = state_str(next_state)

        boundary_env_style = st_s if not next_state_s else f"{st_s} || {next_state_s}"

        # ---------- FIX: boundary change derived from state diffs (NOT from start/end etype) ----------
        boundary_types = ""
        boundary_dates = ""
        if next_state is not None:
            boundary_types = boundary_change_label(st, next_state)
            # the transition moment is the end of the current episode (== start of next)
            boundary_dates = iso(seg["end_dt"])
        else:
            # last episode: no next transition
            boundary_types = ""
            boundary_dates = ""

        se = seg.get("start_event")
        ee = seg.get("end_event")

        kept_eps.append({
            "episode_index": idx,
            "episode_start_dt": seg["start_dt"],
            "episode_end_dt": seg["end_dt"],
            "episode_duration_days": float(seg.get("duration_days", days_between(seg["start_dt"], seg["end_dt"]))),

            "episode_start_commit_sha": (se.get("commit") if se else "") or "",
            "episode_end_commit_sha": (ee.get("commit") if ee else "") or "",

            # state-machine: env_styles is FULL state
            "env_styles": st_s,
            "timeline_env_style": st_s,
            "env_style_mixed": env_style_mixed_from_state(st),

            "boundary_env_style": boundary_env_style,
            "boundary_event_types": boundary_types,
            "boundary_event_dates": boundary_dates,

            "start_event": se,
            "end_event": ee,
        })

    # --- Write episode rows
    for ep in kept_eps:
        episode_rows.append({
            "repo_name": repo_name,
            "full_name": full_name,
            "episode_index": ep["episode_index"],

            "episode_start_utc": iso(ep["episode_start_dt"]),
            "episode_end_utc": iso(ep["episode_end_dt"]),
            "episode_duration_days": f"{ep['episode_duration_days']:.6f}",

            "env_styles": ep["env_styles"],
            "boundary_env_style": ep["boundary_env_style"],
            "timeline_env_style": ep["timeline_env_style"],
            "env_style_mixed": ep["env_style_mixed"],

            # FIXED semantics:
            #   boundary_event_types = added()/removed() from state difference to next episode
            #   boundary_event_dates = transition datetime (end of current == start of next)
            "boundary_event_types": ep["boundary_event_types"],
            "boundary_event_dates": ep["boundary_event_dates"],

            "episode_start_commit_sha": ep["episode_start_commit_sha"],
            "episode_end_commit_sha": ep["episode_end_commit_sha"],
        })

    # ------------------------------------------------------------------
    # Boundary events dataset:
    # include ONLY events actually used by kept segments (start_event and end_event)
    # and assign episode_index by EVENT key (commit, style, etype), not commit alone.
    # ------------------------------------------------------------------
    used_event_keys: Set[Tuple[str, str, str]] = set()

    start_event_to_idx = defaultdict(list)
    end_event_to_idx = defaultdict(list)

    for ep in kept_eps:
        se = ep.get("start_event")
        if se:
            ek = (se["commit"], se["style"], se["etype"])
            used_event_keys.add(ek)
            start_event_to_idx[ek].append(ep["episode_index"])

        ee = ep.get("end_event")
        if ee:
            ek = (ee["commit"], ee["style"], ee["etype"])
            used_event_keys.add(ek)
            end_event_to_idx[ek].append(ep["episode_index"])

    start_keys = set(start_event_to_idx.keys())
    end_keys = set(end_event_to_idx.keys())

    def build_event_state_map(effective: List[Dict]) -> Dict[Tuple[str, str, str], Tuple[str, str]]:
        active: Set[str] = set()
        m: Dict[Tuple[str, str, str], Tuple[str, str]] = {}
        for ev in effective:
            ek = (ev["commit"], ev["style"], ev["etype"])
            before = state_str(fmt_state(active)) if active else ""
            if ev["etype"] == "added":
                active.add(ev["style"])
            else:
                active.discard(ev["style"])
            after = state_str(fmt_state(active)) if active else ""
            m[ek] = (before, after)
        return m

    event_state_map = build_event_state_map(all_effective)

    for ev in all_effective:
        ek = (ev["commit"], ev["style"], ev["etype"])
        if ek not in used_event_keys:
            continue

        is_start = 1 if ek in start_keys else 0
        is_end = 1 if ek in end_keys else 0

        assigned_ep = ""
        if is_start:
            assigned_ep = str(min(start_event_to_idx[ek]))
        elif is_end:
            assigned_ep = str(max(end_event_to_idx[ek]))

        before_state, after_state = event_state_map.get(ek, ("", ""))

        boundary_event_rows.append({
            "repo_name": repo_name,
            "full_name": full_name,
            "timeline_scope": timeline_scope,
            "qa_issue": qa_issue,

            "env_style": ev["style"],
            "event_type": ev["etype"],
            "event_date_utc": iso(ev["dt"]),
            "commit_sha": ev["commit"],

            "from_state": before_state,
            "to_state": after_state,

            "episode_index": assigned_ep,
            "is_episode_start_boundary": is_start,
            "is_episode_end_boundary": is_end,
        })

    # ------------------------------------------------------------------
    # Repo-level summary (Start vs Current status)
    # ------------------------------------------------------------------
    first_ep = kept_eps[0] if kept_eps else None
    current_ep = next(
        (ep for ep in kept_eps if ep["episode_end_dt"] == CUTOFF_DT and ep["episode_end_commit_sha"] == ""),
        None
    )

    repo_level_rows.append({
        "repo_name": repo_name,
        "full_name": full_name,
        "cutoff_date": CUTOFF_ISO,
        "persist_days": PERSIST_DAYS,
        "timeline_scope": timeline_scope,
        "qa_issue": qa_issue,
        "source_json": str(jf),

        "effective_events_count": len(all_effective),
        "raw_state_segments_count": len(raw_segments),
        "kept_state_segments_count": len(kept_segments),

        "first_episode_start_utc": iso(first_ep["episode_start_dt"]) if first_ep else "",
        "first_episode_state": first_ep["env_styles"] if first_ep else "",
        "first_episode_start_commit_sha": first_ep["episode_start_commit_sha"] if first_ep else "",

        "current_episode_start_utc": iso(current_ep["episode_start_dt"]) if current_ep else "",
        "current_episode_state": current_ep["env_styles"] if current_ep else "",
        "current_episode_start_commit_sha": current_ep["episode_start_commit_sha"] if current_ep else "",
        "current_episode_duration_days": (
            f"{days_between(current_ep['episode_start_dt'], CUTOFF_DT):.6f}" if current_ep else ""
        ),
    })


# ------------------------------------------------------------------
# Write outputs
# ------------------------------------------------------------------
episodes_df = pd.DataFrame.from_records(episode_rows)
if not episodes_df.empty:
    episodes_df.sort_values(["repo_name", "episode_index"], inplace=True)

episodes_out_csv = OBS_OUTPUT / "List_Change_Episodes.csv"
episodes_df.to_csv(episodes_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote STATE-MACHINE episode dataset: {episodes_out_csv}")

events_df = pd.DataFrame.from_records(boundary_event_rows)
if not events_df.empty:
    events_df.sort_values(
        ["repo_name", "episode_index", "event_date_utc", "env_style", "event_type"],
        inplace=True
    )

events_out_csv = OBS_OUTPUT / "List_Boundary_Commit_Events.csv"
events_df.to_csv(events_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote STATE-MACHINE boundary-commit dataset: {events_out_csv}")

repo_df = pd.DataFrame.from_records(repo_level_rows)
if not repo_df.empty:
    repo_df.sort_values(["repo_name"], inplace=True)

repo_out_csv = OBS_OUTPUT / "Repo_level_episodes.csv"
repo_df.to_csv(repo_out_csv, index=False, encoding="utf-8")
print(f"[ok] Wrote STATE-MACHINE repo-level summary: {repo_out_csv}")

print("[done] State-machine timeline pipeline finished.")


[info] Found 480 candidate JSON files under Mine dir.
[info] Loaded 480 JSON files successfully.
[info] Selected 480 JSON records for processing (dedup_by_repo=True).
[ok] Wrote STATE-MACHINE episode dataset: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Change_Episodes.csv
[ok] Wrote STATE-MACHINE boundary-commit dataset: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\List_Boundary_Commit_Events.csv
[ok] Wrote STATE-MACHINE repo-level summary: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\2 - RQ2\Observations\Repo_level_episodes.csv
[done] State-machine timeline pipeline finished.
